In [1]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("CapstoneKey")
    

CapstoneKey ········


In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [2]:
# %pip install -qU langchain_community pypdf

In [3]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./Leveraging Machine Learning to Classify Junk Food vs. Regular Food.pdf"
loader = PyPDFLoader(file_path)

In [4]:
docs = loader.load()
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 3163


In [5]:
print(docs[0].page_content[:500])

SIADS Milestone II - Winter, 2025 
Leveraging Machine Learning to Classify Junk Food vs. Regular Food 
Ayan Banerjee, Richard Chalker, Alex DeLoach 
 
Introduction 
The U.S. food supply is heavily dominated by packaged foods and beverages, which contribute 
approximately 75% of daily calorie intake for the population. A significant portion of these 
calories derives from foods commonly perceived as “junk foods.” However, there is no 
universally accepted definition of junk food in academic liter


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 52 sub-documents.


In [7]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [9]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['a8d33b18-899d-4339-91e0-bfd8435acea8', '175d3311-c15e-483c-9ebb-73c2f8c83db4', '5eb7e933-b75c-4090-9002-b3a87d52f0a9']


In [11]:
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")

question = "Summarize the feature importance analysis results for nutrients"

retrieved_docs = vector_store.similarity_search(question)
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = prompt.invoke({"context": docs_content, "question": question})
answer = llm.invoke(prompt)
answer

C:\Users\ayanbanerjee\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langsmith\client.py:253: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


AIMessage(content='The feature importance analysis for nutrients revealed that Sodium, Total Fat, and Sugars are the most significant features for solid food classifications, while Total Fat, Folate, and a specific fatty acid (22:6 n-3) are key for liquids. Removing vital nutrients reduces model accuracy significantly, dropping from the mid-90s to the low 70s for solids and low 80s for liquids. Additionally, carbohydrate content was identified as the fourth most influential feature in liquid food classifications, impacting misclassification results.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 465, 'total_tokens': 570, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_06737a9306', 'finish_reason':